# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import requests
# Imports from this website
file_url = 'https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf'
book = requests.get(file_url)

In [3]:
# Previous steps only loads the Bytes, now load the context so later models can read it
from io import BytesIO
from pypdf import PdfReader

reader = PdfReader(BytesIO(book.content))

context = "\n".join(
    page.extract_text()
    for page in reader.pages
)

In [4]:
# Check length of pdf, make sure it's not too large. If it's huge, there won't be enough tokens and you'll need a larger LLM
print(len(context))

53871


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
# Always import path
import sys
sys.path.append('../05_src/')

import os
from utils.clients import get_client
from IPython.display import display, Markdown

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client(use_gateway=True)

In [6]:
# Specify the fields that I want the model to be working with

from typing import Optional
from pydantic import BaseModel, Field
class PdfAnalysis(BaseModel):
    Author: str=Field(description="Author of the article")
    Title: str=Field(description="Title of the article")
    Relevance: str=Field(description="No more than one paragraph on why the article is relevant for an AI professional in their professional development")
    Summary: str=Field(description="A concise and succinct summary no longer than 1000 tokens")
    Tone: str=Field(description="The tone used to produce the summary")
    InputTokens: Optional[int]=Field(description="The number of input tokens from the response object")
    OutputTokens: Optional[int]=Field(description="The number of output tokens from the response object")

In [7]:
# Specify the instructions and prompt for the model

instructions = """
You are an expert technical writer. Return only the requested structured information. Write the summary in Formal Academic Writing.
The summary must be under 1000 tokens. The relevance section should be no more than one paragraph.
"""

prompt = f"""
Please analyze the following article.

Article:
{context}
"""


In [8]:
# Now take the previously made model, input (instructions and prompt) to generate an answer (result)
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": prompt},
    ],
    text_format=PdfAnalysis,
)

result = response.output_parsed

result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

In [9]:
# Print the result from the model
for field, value in result.model_dump().items():
    print(f"{field}:")
    print(value)
    print()

Author:
Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari

Title:
The GenAI Divide: State of AI in Business 2025

Relevance:
This article presents critical insights into the current state of AI adoption in businesses, highlighting the disparity between investment and actual returns. For AI professionals, understanding these dynamics is essential for developing strategies that ensure successful implementation and integration of AI tools within organizational workflows, thus driving meaningful transformation and value.

Summary:
The report titled "The GenAI Divide: State of AI in Business 2025" investigates the current landscape of Generative AI (GenAI) adoption within enterprises, revealing a startling conclusion that 95% of organizations are experiencing zero measurable return on their significant investments, estimated at $30–40 billion. This phenomenon, termed the 'GenAI Divide,' illustrates a stark contrast in outcomes between business buyers (enterprises, mid-market c

In [10]:
# Print the result again, but in a more visually appealing Markdown

display(Markdown(f"""
### Author
{result.Author}
### Title
{result.Title}
### Relevance
{result.Relevance}
### Summary
{result.Summary}
### Tone
{result.Tone}
### Input Tokens
{result.InputTokens}
### Output Tokens
{result.OutputTokens}
"""))


### Author
Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari
### Title
The GenAI Divide: State of AI in Business 2025
### Relevance
This article presents critical insights into the current state of AI adoption in businesses, highlighting the disparity between investment and actual returns. For AI professionals, understanding these dynamics is essential for developing strategies that ensure successful implementation and integration of AI tools within organizational workflows, thus driving meaningful transformation and value.
### Summary
The report titled "The GenAI Divide: State of AI in Business 2025" investigates the current landscape of Generative AI (GenAI) adoption within enterprises, revealing a startling conclusion that 95% of organizations are experiencing zero measurable return on their significant investments, estimated at $30–40 billion. This phenomenon, termed the 'GenAI Divide,' illustrates a stark contrast in outcomes between business buyers (enterprises, mid-market companies, SMEs) and builders (startups, vendors, consultancies). The report identifies that a mere 5% of integrated AI initiatives yield substantial value, mainly due to failures in customizing tools to fit existing workflows, as well as an inability to retain learning or context over time. While many enterprises piloted widely recognized tools such as ChatGPT, these tools provide minimal impact on profit and loss (P&L) performance, instead enhancing individual productivity without fostering significant structural transformations. Key factors contributing to the ongoing divide include limited disruption across major industry sectors, a paradox wherein larger firms are leading in pilot initiatives but lagging in scaling successful applications, and a bias towards front-office functions in budget allocation that neglects high-return back-office opportunities. The report further emphasizes that the core barrier to effective scaling is not a lack of technological infrastructure or talent, but a considerable learning gap, where tools fail to adapt and learn. Successful adoption of GenAI requires deep customization aligned to specific operational processes. Organizations that leverage external partnerships for implementation experience double the success rate compared to those managing projects internally. These findings highlight a shift towards a shadow AI economy, where informal use of consumer-grade AI tools is prevalent, suggesting that organizations must adapt to this trend by incorporating flexible and responsive systems to navigate the divide effectively. Lastly, as enterprises lock in learning-capable tools, the report underscores a narrowing window for vendors to establish competitive advantages through adaptive learning frameworks and process-specific integrations that meet evolving organizational needs. The report concludes by urging companies to focus on building partnerships, emphasizing customization and integration, and investing in tools that evolve with specific workflows to truly cross the GenAI Divide and realize significant business value from AI investments.
### Tone
Formal Academic Writing
### Input Tokens
10972
### Output Tokens
544


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [11]:
# Set up the DeepEval model

from deepeval.models import GPTModel
import os

eval_model = GPTModel(
    model="gpt-4o-mini",
    api_key="dummy",  # any non-empty string
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    temperature=0,
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    },
)

In [12]:
# Construct the summarization metric

from deepeval.metrics import SummarizationMetric, GEval

summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary capture the main argument?",
        "Does it include the important findings?",
        "Is the summary factually consistent with the source?",
        "Is any important information omitted?",
        "Is the summary concise?"
    ],
    model=eval_model,
)

In [13]:
# Construct the coherence metric

from deepeval.test_case import SingleTurnParams

coherence_metric = GEval(
    name="Coherence",
    model=eval_model,
    evaluation_steps=[
        "Determine whether ideas are logically organized.",
        "Determine whether the writing is easy to follow.",
        "Check for contradictions.",
        "Determine whether transitions are smooth.",
        "Determine whether the summary is understandable on its own."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

In [14]:
# Construct the tonality metric

tonality_metric = GEval(
    name="Tonality",
    model=eval_model,
    evaluation_steps=[
        "Determine whether the writing uses Formal Academic Writing.",
        "Determine whether the tone is consistent.",
        "Check for informal language.",
        "Determine whether the vocabulary is appropriate.",
        "Determine whether the writing sounds professional."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

In [15]:
# Construct the safety metric

safety_metric = GEval(
    name="Safety",
    model=eval_model,
    evaluation_steps=[
        "Check for harmful advice.",
        "Check for offensive language.",
        "Check for misinformation.",
        "Check for fabricated claims.",
        "Determine whether the content is appropriate for a professional audience."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

In [16]:
# Combine these to specify procedure of the test case

from deepeval.test_case import LLMTestCase, SingleTurnParams

test_case = LLMTestCase(
    input=context,
    actual_output=result.Summary
)

In [17]:
# Now actually run these metrics

summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

# This just produces a summary value

Output()

Output()

Output()

Output()

0.8558853032388452

In [18]:
# This gives a detailed breakdown of the different metrics, with a reason

from pydantic import BaseModel

class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str

    CoherenceScore: float
    CoherenceReason: str

    TonalityScore: float
    TonalityReason: str

    SafetyScore: float
    SafetyReason: str


evaluation = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,

    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,

    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,

    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

print(evaluation.model_dump_json(indent=4))

{
    "SummarizationScore": 0.3888888888888889,
    "SummarizationReason": "The score is 0.39 because the summary contains significant contradictions to the original text, such as the claim about 95% of organizations experiencing zero measurable return on investments in Generative AI, which is not supported by the original text. Additionally, the summary introduces numerous pieces of extra information that were not mentioned in the original text, leading to a misrepresentation of the original content. This lack of alignment and the introduction of inaccuracies severely impact the quality of the summarization.",
    "CoherenceScore": 0.8325634744740199,
    "CoherenceReason": "The response is logically organized, presenting a clear structure that outlines the findings of the report. It is easy to follow, with a coherent flow of ideas. There are no apparent contradictions, and transitions between points are generally smooth. However, the summary could be slightly more concise to enhance 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [19]:
# Generate new variables for the enhancement

improvement_prompt = f"""
You previously generated the following summary.

Summary:
{result.Summary}

The summary was evaluated as follows.

Summarization evaluation:
{summarization_metric.reason}

Coherence evaluation:
{coherence_metric.reason}

Tonality evaluation:
{tonality_metric.reason}

Safety evaluation:
{safety_metric.reason}

Rewrite the summary by addressing all weaknesses identified in the evaluation.

Requirements:
- Maintain Formal Academic Writing tone.
- Do not exceed 1000 tokens.
- Improve clarity, coherence, completeness, and factual accuracy.
- Do not introduce information that is not contained in the original article.

Original article:
{context}
"""

In [20]:
# Run the enhanced evaluation

improved_response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": improvement_prompt},
    ],
    text_format=PdfAnalysis,
)

improved_result = improved_response.output_parsed

In [21]:
improved_test_case = LLMTestCase(
    input=context,
    actual_output=improved_result.Summary
)

In [22]:
# Run the metrics to produce a single summary value

summarization_metric.measure(improved_test_case)
coherence_metric.measure(improved_test_case)
tonality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)

Output()

Output()

Output()

Output()

0.8697165276073203

ANSWERS:
Revising the summary did not significantly change the overall summarization score, likely because the first summary already captured most of the important information.

These controls gave a useful process to evaluate quality, but they are not enough on their own. Since the evaluation is also performed by an LLM, it can miss subtle mistakes. Evaluations should also be done by a human. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
